# UASTHN Result Plot
Visualize predicted homography corners from Excel files on satellite maps.

In [ ]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from PIL import Image

# ==============================
# CONFIG
# ==============================
PROJECT_ROOT = os.getcwd()
PRED_EXCEL = "js_excels/sing-two-orig.xlsx"
REF_EXCEL = "js_excels/sing-one-orig.xlsx"  # optional
GT_EXCEL = "js_excels/hemat_gt.xlsx"  # GT excel file

USE_REFERENCE = True
USE_GROUND_TRUTH = True

# Toggle which overlays to show
SHOW_PRED = True      # Red
SHOW_REF = True       # Blue  
SHOW_GT = True        # Green

NUM_IMAGES = None      # None => all rows
GRID_COLS = 6
ALPHA_BLEND = 1    # 1.0 => keep only warped thermal pixels

SAVE_FIGURE = False
SAVE_PATH = "js_excels/overlay_grid.png"
FIG_DPI = 180

def resolve_img_path(path_text):
    path_text = str(path_text)
    if os.path.exists(path_text):
        return path_text
    candidate = os.path.join(PROJECT_ROOT, path_text)
    if os.path.exists(candidate):
        return candidate
    return None


def get_filename_from_path(path_text):
    """Extract filename from path without extension"""
    return os.path.splitext(os.path.basename(str(path_text)))[0]


def draw_quad(image, row, color, thickness, label=None):
    quad = np.array(
        [
            [row["x1"], row["y1"]],  # TL
            [row["x2"], row["y2"]],  # TR
            [row["x4"], row["y4"]],  # BR
            [row["x3"], row["y3"]],  # BL
        ],
        dtype=np.int32,
    )
    cv2.polylines(image, [quad], isClosed=True, color=color, thickness=thickness)
    if label is not None:
        center = np.mean(quad, axis=0).astype(np.int32)
        cv2.putText(
            image,
            label,
            (int(center[0]) - 30, int(center[1])),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2,
            cv2.LINE_AA,
        )


def calculate_distance(point1, point2):
    """Calculate Euclidean distance between two points"""
    return np.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)


def get_corner_points(row):
    """Extract 4 corner points from a row"""
    return np.array([
        [row["x1"], row["y1"]],
        [row["x2"], row["y2"]],
        [row["x3"], row["y3"]],
        [row["x4"], row["y4"]]
    ])


def get_center_point(row):
    """Calculate center point from 4 corners"""
    corners = get_corner_points(row)
    return np.mean(corners, axis=0)


def compute_distances(row1, row2):
    """Compute distances between two sets of points (center + 4 corners)"""
    # Center points
    center1 = get_center_point(row1)
    center2 = get_center_point(row2)
    center_dist = calculate_distance(center1, center2)
    
    # Corner points
    corners1 = get_corner_points(row1)
    corners2 = get_corner_points(row2)
    
    corner_dists = [calculate_distance(corners1[i], corners2[i]) for i in range(4)]
    
    return {
        'center_dist': center_dist,
        'corner_dists': corner_dists,
        'avg_corner_dist': np.mean(corner_dists),  # MACE = Mean Absolute Corner Error
        'max_corner_dist': np.max(corner_dists),
        'min_corner_dist': np.min(corner_dists)
    }


# ==============================
# LOAD EXCEL FILES
# ==============================
pred_excel_path = PRED_EXCEL if os.path.exists(PRED_EXCEL) else os.path.join(PROJECT_ROOT, PRED_EXCEL)
if not os.path.exists(pred_excel_path):
    raise FileNotFoundError(f"Prediction Excel not found: {pred_excel_path}")

pred_df = pd.read_excel(pred_excel_path)
ref_df = None
gt_df = None

ref_excel_path = REF_EXCEL if os.path.exists(REF_EXCEL) else os.path.join(PROJECT_ROOT, REF_EXCEL)
if USE_REFERENCE and os.path.exists(ref_excel_path):
    ref_df = pd.read_excel(ref_excel_path)

gt_excel_path = GT_EXCEL if os.path.exists(GT_EXCEL) else os.path.join(PROJECT_ROOT, GT_EXCEL)
if USE_GROUND_TRUTH and os.path.exists(gt_excel_path):
    gt_df = pd.read_excel(gt_excel_path)

if NUM_IMAGES is not None:
    pred_df = pred_df.head(NUM_IMAGES)
    if ref_df is not None:
        ref_df = ref_df.head(NUM_IMAGES)
    if gt_df is not None:
        gt_df = gt_df.head(NUM_IMAGES)

required_cols = ["x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4", "sat", "th"]
missing = [c for c in required_cols if c not in pred_df.columns]
if missing:
    raise KeyError(f"Missing columns in prediction Excel: {missing}")

# ==============================
# DISTANCE REPORTING TABLE
# ==============================
distance_data = []

for idx, row_pred in pred_df.iterrows():
    row_ref = ref_df.iloc[idx] if ref_df is not None and idx < len(ref_df) else None
    row_gt = gt_df.iloc[idx] if gt_df is not None and idx < len(gt_df) else None
    
    row_data = {'img_idx': idx}
    
    # GT vs PRED
    if row_gt is not None:
        gt_pred = compute_distances(row_gt, row_pred)
        row_data['pred_ce'] = gt_pred['center_dist']  # CE = Center Error
        row_data['pred_mace'] = gt_pred['avg_corner_dist']  # MACE = Mean Absolute Corner Error
        row_data['pred_max_corner'] = gt_pred['max_corner_dist']
        row_data['pred_min_corner'] = gt_pred['min_corner_dist']
    
    # GT vs REF
    if row_gt is not None and row_ref is not None:
        gt_ref = compute_distances(row_gt, row_ref)
        row_data['ref_ce'] = gt_ref['center_dist']
        row_data['ref_mace'] = gt_ref['avg_corner_dist']
        row_data['ref_max_corner'] = gt_ref['max_corner_dist']
        row_data['ref_min_corner'] = gt_ref['min_corner_dist']
    
    # REF vs PRED
    if row_ref is not None:
        ref_pred = compute_distances(row_ref, row_pred)
        row_data['ref_pred_ce'] = ref_pred['center_dist']
        row_data['ref_pred_mace'] = ref_pred['avg_corner_dist']
        row_data['ref_pred_max_corner'] = ref_pred['max_corner_dist']
        row_data['ref_pred_min_corner'] = ref_pred['min_corner_dist']
    
    distance_data.append(row_data)

# Create distance dataframe
dist_df = pd.DataFrame(distance_data)

# Print formatted table in console
print("\n" + "="*100)
print("DISTANCE REPORT (in pixels)")
print("="*100)
print("CE = Center Error | MACE = Mean Absolute Corner Error")
print("-"*100)

# Prepare table data
headers = ["Comparison", "MACE Avg", "MACE Std", "MACE Max", "CE Avg", "CE Std", "CE Max"]

# GT vs PRED
if 'pred_mace' in dist_df.columns:
    table_data = []
    table_data.append([
        "GT vs PRED",
        f"{dist_df['pred_mace'].mean():.2f}",
        f"{dist_df['pred_mace'].std():.2f}",
        f"{dist_df['pred_max_corner'].max():.2f}",
        f"{dist_df['pred_ce'].mean():.2f}",
        f"{dist_df['pred_ce'].std():.2f}",
        f"{dist_df['pred_ce'].max():.2f}"
    ])

# GT vs REF
if 'ref_mace' in dist_df.columns:
    table_data.append([
        "GT vs REF",
        f"{dist_df['ref_mace'].mean():.2f}",
        f"{dist_df['ref_mace'].std():.2f}",
        f"{dist_df['ref_max_corner'].max():.2f}",
        f"{dist_df['ref_ce'].mean():.2f}",
        f"{dist_df['ref_ce'].std():.2f}",
        f"{dist_df['ref_ce'].max():.2f}"
    ])

# REF vs PRED
if 'ref_pred_mace' in dist_df.columns:
    table_data.append([
        "REF vs PRED",
        f"{dist_df['ref_pred_mace'].mean():.2f}",
        f"{dist_df['ref_pred_mace'].std():.2f}",
        f"{dist_df['ref_pred_max_corner'].max():.2f}",
        f"{dist_df['ref_pred_ce'].mean():.2f}",
        f"{dist_df['ref_pred_ce'].std():.2f}",
        f"{dist_df['ref_pred_ce'].max():.2f}"
    ])

# Print table
if table_data:
    # Calculate column widths
    col_widths = [len(h) for h in headers]
    for row in table_data:
        for i, cell in enumerate(row):
            col_widths[i] = max(col_widths[i], len(cell))
    
    # Print header
    header_line = " | ".join(h.ljust(col_widths[i]) for i, h in enumerate(headers))
    print(header_line)
    print("-" * len(header_line))
    
    # Print rows
    for row in table_data:
        row_line = " | ".join(cell.ljust(col_widths[i]) for i, cell in enumerate(row))
        print(row_line)
    
    print("="*100)
    print(f"Total images analyzed: {len(dist_df)}")
    print("="*100 + "\n")

    print("="*100 + "\n")

# ==============================
# RENDER OVERLAYS
# ==============================
results = []

for idx, row_pred in pred_df.iterrows():
    sat_path = resolve_img_path(row_pred["sat"])
    th_path = resolve_img_path(row_pred["th"])

    if sat_path is None or th_path is None:
        print(f"[SKIP] Missing image at row {idx}")
        continue

    big_map = np.array(Image.open(sat_path).convert("RGB"))
    small_map = np.array(Image.open(th_path).convert("RGB"))

    h, w = small_map.shape[:2]
    src_pts = np.array(
        [
            [0, 0],
            [w - 1, 0],
            [0, h - 1],
            [w - 1, h - 1],
        ],
        dtype=np.float32,
    )

    dst_pts = np.array(
        [
            [row_pred["x1"], row_pred["y1"]],
            [row_pred["x2"], row_pred["y2"]],
            [row_pred["x3"], row_pred["y3"]],
            [row_pred["x4"], row_pred["y4"]],
        ],
        dtype=np.float32,
    )

    H, _ = cv2.findHomography(src_pts, dst_pts)
    if H is None:
        print(f"[FAIL] Homography failed at row {idx}")
        continue

    warped = cv2.warpPerspective(
        small_map,
        H,
        (big_map.shape[1], big_map.shape[0]),
    )

    overlay = big_map.copy()
    mask = np.any(warped != 0, axis=2)
    if ALPHA_BLEND >= 1.0:
        overlay[mask] = warped[mask]
    else:
        overlay[mask] = (
            ALPHA_BLEND * warped[mask] + (1.0 - ALPHA_BLEND) * overlay[mask]
        ).astype(np.uint8)

    # Draw GT (green) - draw first so it's at the bottom
    if SHOW_GT and gt_df is not None and idx < len(gt_df):
        row_gt = gt_df.iloc[idx]
        draw_quad(overlay, row_gt, color=(0, 255, 0), thickness=4, label="GT")
    
    # Draw REF (blue)
    if SHOW_REF and ref_df is not None and idx < len(ref_df):
        row_ref = ref_df.iloc[idx]
        draw_quad(overlay, row_ref, color=(0, 0, 255), thickness=3, label="Ref")
    
    # Draw PRED (red) - draw last so it's on top
    if SHOW_PRED:
        draw_quad(overlay, row_pred, color=(255, 0, 0), thickness=2, label="Pred")

    # Get metrics for this image
    pred_ce = dist_df.iloc[idx]['pred_ce'] if 'pred_ce' in dist_df.columns and idx < len(dist_df) else None
    pred_mace = dist_df.iloc[idx]['pred_mace'] if 'pred_mace' in dist_df.columns and idx < len(dist_df) else None
    ref_ce = dist_df.iloc[idx]['ref_ce'] if 'ref_ce' in dist_df.columns and idx < len(dist_df) else None
    ref_mace = dist_df.iloc[idx]['ref_mace'] if 'ref_mace' in dist_df.columns and idx < len(dist_df) else None
    
    # Create title with CE and MACE for both PRED and REF
    title_text = f"{idx}: {os.path.basename(th_path)}"
    if pred_ce is not None and pred_mace is not None:
        title_text += f"\nPRED CE: {pred_ce:.1f}px | MACE: {pred_mace:.1f}px"
    if ref_ce is not None and ref_mace is not None:
        title_text += f"\nREF  CE: {ref_ce:.1f}px | MACE: {ref_mace:.1f}px"
    
    results.append((overlay, title_text))

if len(results) == 0:
    raise RuntimeError("No overlays were rendered. Check Excel paths and image paths.")

# ==============================
# CREATE FIGURE WITH FIXED SPACING
# ==============================
rows = math.ceil(len(results) / GRID_COLS)

# Use subplots with proper spacing
fig, axes = plt.subplots(rows, GRID_COLS, figsize=(5 * GRID_COLS, 5 * rows))
fig.subplots_adjust(top=0.92, hspace=0.15, wspace=0.1)

# Flatten axes for easy indexing
if rows == 1 and GRID_COLS == 1:
    axes = np.array([axes])
axes_flat = axes.flatten() if rows > 1 or GRID_COLS > 1 else axes

# Plot images
for i, (img, title) in enumerate(results):
    ax = axes_flat[i]
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

# Hide empty subplots
for i in range(len(results), len(axes_flat)):
    axes_flat[i].axis("off")

# Prepare legend elements
legend_elements = []

# Get file names from Excel paths
pred_name = get_filename_from_path(PRED_EXCEL)
ref_name = get_filename_from_path(REF_EXCEL)
gt_name = get_filename_from_path(GT_EXCEL)

if SHOW_GT:
    legend_elements.append(Patch(facecolor='green', edgecolor='green', 
                                label=f'GT: {gt_name}', linewidth=3))
if SHOW_REF:
    legend_elements.append(Patch(facecolor='blue', edgecolor='blue', 
                                label=f'REF: {ref_name}', linewidth=3))
if SHOW_PRED:
    legend_elements.append(Patch(facecolor='red', edgecolor='red', 
                                label=f'PRED: {pred_name}', linewidth=2))

legend_elements.append(Patch(facecolor='none', edgecolor='none', 
                            label='CE: Center Error | MACE: Mean Absolute Corner Error'))

# Add legend at the top of the figure
fig.legend(handles=legend_elements, 
           loc='upper center',
           bbox_to_anchor=(0.5, 0.98),
           ncol=min(4, len(legend_elements)), 
           fontsize=9,
           frameon=True, 
           fancybox=True, 
           shadow=True)

# Save figure if requested
if SAVE_FIGURE:
    out_path = SAVE_PATH if os.path.isabs(SAVE_PATH) else os.path.join(PROJECT_ROOT, SAVE_PATH)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    print(f"Saved figure to: {out_path}")

plt.show()
print(f"Rendered {len(results)} overlays.")
